Improvement: Added LSTM layer for temporal relation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go

import utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
def dtw_distance(seq1, seq2):
    n, m = len(seq1), len(seq2)
    dtw_matrix = np.zeros((n+1, m+1))
    for i in range(1, n+1):
        dtw_matrix[i, 0] = np.inf
    for i in range(1, m+1):
        dtw_matrix[0, i] = np.inf
    dtw_matrix[0, 0] = 0

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(seq1[i-1] - seq2[j-1])
            last_min = min(dtw_matrix[i-1, j],    # insertion
                           dtw_matrix[i, j-1],    # deletion
                           dtw_matrix[i-1, j-1])  # match
            dtw_matrix[i, j] = cost + last_min
    return dtw_matrix[n, m]

# Data Loading

In [3]:
kmedoid_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_medoids.csv')
medoid_to_spike_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_medoid_to_spike.csv')
test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')

print(kmedoid_df.shape, medoid_to_spike_df.shape, test_df.shape)

(344618, 19) (33916616, 19) (1682533, 19)


In [4]:
medoid_config_file = 'Config/medoid_config.json'
m2s_config_file = 'Config/m2s_config.json'

with open(medoid_config_file, 'r') as f:
    medoid_config = json.load(f)

with open(m2s_config_file, 'r') as f:
    m2s_config = json.load(f)

model_medoid = network.m_VAE(**medoid_config).to(device)
model_medoid.load_state_dict(torch.load('../../Models/medoid_vae_best.pt', map_location=device))

model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s.load_state_dict(torch.load('../../Models/m2s_vae_test.pt', map_location=device))

model_medoid.eval()
model_m2s.eval()

clear_output = True

## See how the Medoid_VAE fit the medoid_df

In [5]:
ids = kmedoid_df['LCLid'].unique()
np.random.seed(0)
id = np.random.choice(ids, 1)
temp_df = kmedoid_df[kmedoid_df['LCLid'] == id[0]]

weather, cluster, time, statistical, spike,spike_mag, noise, real_energy = utils.get_information(sequence_length, temp_df)

h = model_medoid.lstm_decoder(noise, weather, cluster, time)

fake_energy = h.squeeze().detach().cpu().numpy()
real_energy = real_energy.squeeze()

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(len(real_energy)), y=real_energy, mode='lines', name='Real'))
fig.add_trace(go.Scatter(x=np.arange(len(fake_energy)), y=fake_energy, mode='lines', name='Synthetic', line=dict(color='orange')))

fig.update_layout(title='Real vs Synthetic Energy in the k-Medoid Set (Training set)', xaxis_title='Index', yaxis_title='Energy')


# See how the Regular-to-Spike fit the m2s_df, which the model is trained on

In [6]:
ids = medoid_to_spike_df['LCLid'].unique()
# np.random.seed(5)
id = np.random.choice(ids, 1)
temp_df = medoid_to_spike_df[medoid_to_spike_df['LCLid'] == id[0]]

weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(200, temp_df)

latent_space = model_medoid.lstm_decoder(noise, weather, cluster, time)

latent_space = model_m2s.lstm_encoder(latent_space)
mu, logvar = latent_space.chunk(2, dim=2)
z = model_m2s.reparameterize(mu, logvar)
h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

fake_energy = h.squeeze().detach().cpu().numpy()
real_energy = real_energy.squeeze()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(real_energy)), 
    y=real_energy, 
    mode='lines', 
    name='Real'))

fig.add_trace(go.Scatter(
    x=np.arange(len(fake_energy)), 
    y=fake_energy, 
    mode='lines', 
    name='Synthetic', 
    line=dict(color='orange', width = 1.5)))

fig.update_layout(
    title='Real vs Synthetic Energy in the Medoid to Regular Set (Training set)', 
    xaxis_title='Time', 
    yaxis_title='Energy')

# See how these three networks work together in never seen households

In [8]:
ids = test_df['LCLid'].unique()
np.random.seed(3)
id = np.random.choice(ids, 1)
temp_df = test_df[test_df['LCLid'] == id[0]]

weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(200, temp_df)
print('Weather shape:', weather.shape)
print('Cluster shape:', cluster.shape)
print('Time shape:', time.shape)
print('Statistical shape:', statistical.shape)
print('Spike type shape:', spike_type.shape)
print('Spike magnitude shape:', spike_magnitude.shape)
print('Noise shape:', noise.shape)
print('Real energy shape:', real_energy.shape)
h = model_medoid.lstm_decoder(noise, weather, cluster, time)
print(h.shape)

latent_space = model_m2s.lstm_encoder(h)
mu, logvar = latent_space.chunk(2, dim=2)
z = model_m2s.reparameterize(mu, logvar)
h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

fake_energy = h.squeeze().detach().cpu().numpy()
real_energy = real_energy.squeeze()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(real_energy)), 
    y=real_energy, 
    mode='lines', 
    name='Real'))

fig.add_trace(go.Scatter(
    x=np.arange(len(fake_energy)), 
    y=fake_energy, 
    mode='lines', 
    name='Synthetic', 
    line=dict(color='orange')))

fig.update_layout(
    title='Real vs Synthetic Energy in the Test Set', 
    xaxis_title='Timestamp', 
    yaxis_title='Energy')

fig.update_layout(
    font=dict(size=18),          # global font (axes, title, etc.)
    legend=dict(font=dict(size=18)),
)

fig.update_layout(
    width=1000,   # make it less wide (try 600–900)
    height=450   # optional
)

Weather shape: torch.Size([1, 200, 3])
Cluster shape: torch.Size([1, 200, 1])
Time shape: torch.Size([1, 200, 4])
Statistical shape: torch.Size([1, 200, 6])
Spike type shape: torch.Size([1, 200, 1])
Spike magnitude shape: torch.Size([1, 200, 1])
Noise shape: torch.Size([1, 200, 64])
Real energy shape: (200, 1)
torch.Size([1, 200, 1])


In [8]:
mse = []
srcc = []
corr = []
dtw = []
cosine_similarity_score = []

for i in range(200):
    id = np.random.choice(ids, 1)
    temp_df = test_df[test_df['LCLid'] == id[0]]

    weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(200, temp_df)

    h = model_medoid.lstm_decoder(noise, weather, cluster, time)

    latent_space = model_m2s.lstm_encoder(h)
    mu, logvar = latent_space.chunk(2, dim=2)
    z = model_m2s.reparameterize(mu, logvar)
    h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

    synthetic_profile = h.squeeze().detach().cpu().numpy()
    real_energy = real_energy.squeeze()

    mse.append(utils.mean_squared_error(synthetic_profile, real_energy))
    srcc.append(utils.spearman_correlation(synthetic_profile, real_energy))
    corr.append(utils.pearson_correlation(synthetic_profile, real_energy))
    dtw.append(utils.dtw_distance(synthetic_profile, real_energy))
    cosine_similarity_score.append(utils.cosine_similarity_score(synthetic_profile, real_energy))

print(f'MSE: {np.mean(mse)}')
print(f'SRCC: {np.mean(srcc)}')
print(f'Correlation: {np.mean(corr)}')
print(f'DTW: {np.mean(dtw)}')
print(f'Cosine Similarity Score: {np.mean(cosine_similarity_score)}')

MSE: 0.018732650833935466
SRCC: 0.5801648750265826
Correlation: 0.7824163189676417
DTW: 1.081096430929779
Cosine Similarity Score: 0.9301630566043845


# Some Enhancement

In [9]:
temp_df = test_df[test_df['LCLid'] == id[0]]

weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(200, temp_df)

h = model_medoid.lstm_decoder(noise, weather, cluster, time)

latent_space = model_m2s.lstm_encoder(h)
mu, logvar = latent_space.chunk(2, dim=2)
z = model_m2s.reparameterize(mu, logvar)
h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

synthetic_energy = h.squeeze().detach().cpu().numpy()
gradient = statistical.squeeze().detach().cpu().numpy()[0][-1]
min_energy = statistical.squeeze().detach().cpu().numpy()[0][3]
enhanced_synthetic_energy = utils.enhanced_energy_profile(spike_type, synthetic_energy, gradient, min_energy)

real_energy = real_energy.squeeze()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(real_energy)), 
    y=real_energy, 
    mode='lines', 
    name='Real'))

fig.add_trace(go.Scatter(
    x=np.arange(len(synthetic_energy)), 
    y=synthetic_energy, 
    mode='lines', 
    name='Synthetic', 
    line=dict(color='orange')))

fig.add_trace(go.Scatter(
    x=np.arange(len(enhanced_synthetic_energy)), 
    y=enhanced_synthetic_energy, 
    mode='lines', 
    name='Enhanced Synthetic', 
    line=dict(color='green')))

fig.update_layout(
    title='Real vs Synthetic Energy in the Test Set', 
    xaxis_title='Time', 
    yaxis_title='Energy')

In [10]:
mse = []
srcc = []
corr = []
dtw = []
cosine_similarity_score = []

ids = test_df['LCLid'].unique()

for i in range(200):
    id = np.random.choice(ids, 1)
    temp_df = test_df[test_df['LCLid'] == id[0]]

    weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(200, temp_df)

    h = model_medoid.lstm_decoder(noise, weather, cluster, time)

    latent_space = model_m2s.lstm_encoder(h)
    mu, logvar = latent_space.chunk(2, dim=2)
    z = model_m2s.reparameterize(mu, logvar)
    h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

    synthetic_profile = h.squeeze().detach().cpu().numpy()
    gradient = statistical.squeeze().detach().cpu().numpy()[0][-1]
    min_energy = statistical.squeeze().detach().cpu().numpy()[0][3]
    enhanced_synthetic_profile = utils.enhanced_energy_profile(spike_type, synthetic_profile, gradient, min_energy)
    real_energy = real_energy.squeeze()

    mse.append(utils.mean_squared_error(enhanced_synthetic_profile, real_energy))
    srcc.append(utils.spearman_correlation(enhanced_synthetic_profile, real_energy))
    corr.append(utils.pearson_correlation(enhanced_synthetic_profile, real_energy))
    dtw.append(utils.dtw_distance(enhanced_synthetic_profile, real_energy))
    cosine_similarity_score.append(utils.cosine_similarity_score(enhanced_synthetic_profile, real_energy))

print(f'MSE: {np.mean(mse)}')
print(f'SRCC: {np.mean(srcc)}')
print(f'Correlation: {np.mean(corr)}')
print(f'DTW: {np.mean(dtw)}')
print(f'Cosine Similarity Score: {np.mean(cosine_similarity_score)}')

MSE: 0.01918914317881075
SRCC: 0.5870228694972313
Correlation: 0.7987622313772695
DTW: 1.1059489528927013
Cosine Similarity Score: 0.9267882043971007
